# Section 2 — Foundry-Backed Agents + DevUI

## Storyline

PolicyPal works in Section 1, but every iteration loop today looks like: edit code → run cell → squint at the printed text → repeat. When you hand PolicyPal to an Athora Netherlands developer or domain reviewer, they want a **chat UI** they can poke at, plus a way to **inspect tool calls** when something goes wrong.

**DevUI** is the local, framework-provided debug surface. It serves any `Agent` (or `Workflow`) over an OpenAI-compatible HTTP API and gives you a browser UI on `http://localhost:8090`. No deployment, no Foundry portal needed — just `serve(entities=[...])`.

## What you'll do

| Step | Concept | Why it matters for Athora Netherlands |
|------|---------|---------------------------|
| 1 | Confirm Foundry inference connection | Athora Netherlands cannot use the hosted agent service. We connect via `FoundryChatClient` (model inference). |
| 2 | Wrap PolicyPal + tools as a `serve()`-able entity | One file, one command, the dev gets a real chat UI. |
| 3 | Inspect a tool-call trace in DevUI | When PolicyPal calls `get_policy()` with the wrong policy number, you'll see it. |

## Why DevUI matters in this workshop

Once we move into workflows (Section 3) and observability (Section 5), you'll want the **same tool** to debug both. DevUI handles agents *and* workflows — same endpoint, same UI.

## A note on running DevUI

`serve()` is a **blocking** call (it runs a uvicorn server). Inside a Jupyter cell that would wedge the kernel. So for this section we'll:

1. Build PolicyPal in this notebook so you understand the code.
2. Save the same code to a small `policypal_devui.py` file.
3. Run it from a **terminal** with `python policypal_devui.py`.

DevUI will open in your browser at `http://localhost:8090`.

## DevUI-ready PolicyPal tools

First we define the same small policy tools from Section 1. DevUI will show each model-selected tool call, the arguments, and the result. Reference: [`microsoft/agent-framework — devui samples`](https://github.com/microsoft/agent-framework/tree/main/python/samples/devui).

In [1]:
import os
from typing import Annotated
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from pydantic import Field

load_dotenv()

# Mock Athora Netherlands policy data — same holders we used in Section 1
POLICIES = {
    "NL-2031-887": {
        "holder": "Jan de Vries",
        "product": "Pension - defined contribution",
        "balance_eur": 78_420.55,
        "monthly_contribution_eur": 350.00,
        "inception_date": "2014-06-01",
    },
    "NL-4408-552": {
        "holder": "Sanne Bakker",
        "product": "Life insurance - term",
        "balance_eur": 0.0,
        "monthly_contribution_eur": 42.50,
        "inception_date": "2019-11-15",
    },
}

@tool(approval_mode="never_require")
def get_policy(policy_number: Annotated[str, Field(description="Athora Netherlands policy number, e.g. NL-2031-887")]) -> str:
    """Look up an Athora Netherlands policy by its number."""
    p = POLICIES.get(policy_number.upper())
    if not p:
        return f"No policy found for {policy_number}."
    return f"Policy {policy_number}: {p}"

@tool(approval_mode="never_require")
def project_pension_balance(
    policy_number: Annotated[str, Field(description="Pension policy number")],
    years: Annotated[int, Field(description="Years to project forward")],
) -> str:
    """Rough projection assuming 4% annual growth on current balance + monthly contribution."""
    p = POLICIES.get(policy_number.upper())
    if not p or "balance_eur" not in p:
        return f"No pension found for {policy_number}."
    balance = p["balance_eur"]
    monthly = p["monthly_contribution_eur"]
    for _ in range(years):
        balance = balance * 1.04 + monthly * 12
    return f"Projected balance after {years} years: ~ EUR {balance:,.0f}"

INSTRUCTIONS = (
    "You are PolicyPal, an internal assistant for Athora Netherlands customer-service representatives. "
    "Be concise, factual, and never invent policy data — always use the tools. "
    "For pension projections, remind the rep that figures are illustrative and not advice."
)

# This `policypal` object is what we'll hand to DevUI.
policypal = Agent(
    client=FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["FOUNDRY_MODEL"],
        credential=AzureCliCredential(),
    ),
    name="policypal",
    description="Athora Netherlands customer-service assistant (workshop demo).",
    instructions=INSTRUCTIONS,
    tools=[get_policy, project_pension_balance],
)
print("PolicyPal ready:", policypal.name)

PolicyPal ready: policypal


### What just happened

We built **the same PolicyPal as Section 1** but in a single block, suitable for handing to `serve()`. Notice:

- We pass an explicit `description` — DevUI shows it in the agent picker.
- We did **not** call `agent.run()` here. DevUI calls it for us per chat turn.
- All credentials still come from `.env` and `az login` — no new auth model.

## Write the DevUI launcher script

Run the cell below to write `policypal_devui.py` next to this notebook. Then in a terminal:

```bash
az login
python policypal_devui.py
```

DevUI will open at `http://localhost:8090`.

In [4]:
from pathlib import Path

LAUNCHER = '''# Auto-generated by Section 2 of the Athora Netherlands workshop.
import os
from typing import Annotated
from agent_framework import Agent, tool
from agent_framework.devui import serve
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from pydantic import Field

load_dotenv()

POLICIES = {
    "NL-2031-887": {"holder": "Jan de Vries", "product": "Pension - defined contribution",
                     "balance_eur": 78_420.55, "monthly_contribution_eur": 350.00,
                     "inception_date": "2014-06-01"},
    "NL-4408-552": {"holder": "Sanne Bakker", "product": "Life insurance - term",
                     "balance_eur": 0.0, "monthly_contribution_eur": 42.50,
                     "inception_date": "2019-11-15"},
}

@tool(approval_mode="never_require")
def get_policy(policy_number: Annotated[str, Field(description="Athora Netherlands policy number")]) -> str:
    """Look up an Athora Netherlands policy by its number."""
    p = POLICIES.get(policy_number.upper())
    return f"Policy {policy_number}: {p}" if p else f"No policy found for {policy_number}."

@tool(approval_mode="never_require")
def project_pension_balance(policy_number: Annotated[str, Field(description="Pension policy number")],
                             years: Annotated[int, Field(description="Years to project forward")]) -> str:
    """Rough projection assuming 4% annual growth + monthly contribution."""
    p = POLICIES.get(policy_number.upper())
    if not p or "balance_eur" not in p:
        return f"No pension found for {policy_number}."
    bal, m = p["balance_eur"], p["monthly_contribution_eur"]
    for _ in range(years):
        bal = bal * 1.04 + m * 12
    return f"Projected balance after {years} years: ~ EUR {bal:,.0f}"

policypal = Agent(
    client=FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["FOUNDRY_MODEL"],
        credential=AzureCliCredential(),
    ),
    name="policypal",
    description="Athora Netherlands customer-service assistant (workshop demo).",
    instructions=(
        "You are PolicyPal, an internal assistant for Athora Netherlands customer-service reps. "
        "Be concise, factual, and never invent policy data — always use the tools."
    ),
    tools=[get_policy, project_pension_balance],
)

if __name__ == "__main__":
    print("DevUI starting at http://localhost:8090 ...")
    serve(entities=[policypal], port=8090, auto_open=True,instrumentation_enabled=True)
'''
Path("policypal_devui.py").write_text(LAUNCHER, encoding="utf-8")
print("Wrote policypal_devui.py — now run: python policypal_devui.py")

Wrote policypal_devui.py — now run: python policypal_devui.py


### What you should see in DevUI

- **Agent picker** in the sidebar showing `policypal`.
- A chat box. Try: *"What's on policy NL-2031-887?"*
- A **Trace** / **Tool Calls** panel — for each turn you'll see:
  - The system + user messages sent to the model.
  - Tool calls the model made (`get_policy({"policy_number": "NL-2031-887"})`).
  - The tool's return value.
  - The model's final text.

As you can see, by adding serve() agent framework handles the magic for you!

## What to inspect in DevUI

Use a representative rep prompt such as: *"Can you project Jan de Vries' pension 10 years out? Use policy NL-2031-887."* In the trace panel, inspect the selected tool, the `policy_number` and `years` arguments, the returned projection, and the final wording. Then try `NL-9999-000` to see how a tool error becomes a safe user-facing response.

## Recap and what's next

- A single `Agent` object is enough to run end-to-end in DevUI — no deployment, no portal, no hosted-agent service.
- DevUI surfaces the **tool-call trace** which is your single most useful debugging artifact.
- The same `serve()` function will accept a `Workflow` in **Section 3**, where PolicyPal becomes one node in a multi-step claim-triage pipeline.
